In [2]:
import xarray as xr
import rasterio
from rasterio.transform import from_origin
import numpy as np

In [3]:
from rasterio.crs import CRS

In [4]:
import os

In [5]:
import netCDF4
import rioxarray
import rasterio

In [6]:
base = os.path.join(os.getcwd(),'..')

In [7]:
data = os.path.join(base,'data(LPJmL)','new_run')

In [8]:
for r,d,f in os.walk(data):
    for fl in f:
        if fl.endswith('.nc') and 'lai' in fl:
            print(fl)
            fn = os.path.join(r,fl)


pft_lai.nc


In [9]:
ls = ("temperate_cereals","rice","maize","tropical cereals","pulses","temperate roots",
             "tropical roots","sunflower","soybeans","groundnuts","rapeseed","sugarcane", 
      "barley","cotton","wheat2","rice2", "rice3","others","grasses","biofuels1","biofuels2")


In [10]:
# ds = netCDF4.Dataset(fn)
ds = xr.open_dataset(fn, decode_times=False)

In [11]:
ds

<xarray.Dataset> Size: 184GB
Dimensions:    (pft: 53, time: 120, bnds: 2, lat: 1676, lon: 4320)
Coordinates:
  * pft        (pft) int32 212B 1 2 3 4 5 6 7 8 9 ... 45 46 47 48 49 50 51 52 53
  * time       (time) float64 960B 3.98e+04 3.983e+04 ... 4.339e+04 4.342e+04
  * lat        (lat) float64 13kB -55.96 -55.87 -55.79 ... 83.46 83.54 83.63
  * lon        (lon) float64 35kB -180.0 -179.9 -179.8 ... 179.8 179.9 180.0
Dimensions without coordinates: bnds
Data variables:
    NamePFT    (pft) <U38 8kB ...
    time_bnds  (time, bnds) float64 2kB ...
    lat_bnds   (lat, bnds) float64 27kB ...
    lon_bnds   (lon, bnds) float64 69kB ...
    LAI        (time, pft, lat, lon) float32 184GB ...
Attributes:
    title:        simulation_global_dashb_test2
    source:       LPJmL C Version 5.10.1
    GIT_repo:     https://gitlab.pik-potsdam.de/lpjml/wur_team/lpjml-wur.git
    GIT_hash:     77c3b07d1e88121358c6e1e135ee10aafd181de3
    history:      Mon Mar  2 14:11:00 2026: /home/WUR/biema005/LPJmL5_git/bin...
    institution:  Potsdam Institute for Climate Impact Research
    contact:      
    comment:

In [12]:
var_name = 'LAI'
ds = xr.open_dataset(fn, decode_times=False, engine='netcdf4')
da = ds[var_name]


evap = ds[var_name]    
pft_names = ds["NamePFT"].values  # strings

In [13]:
len(pft_names)

53

In [14]:
c=0
for d in pft_names:
    c+=1
    if 'maize' in d:
        print(d,c)
    

rainfed maize 14
irrigated maize 35


In [16]:
# fn = os.path.join(r,'cft_evap.nc')
nc_file = fn
var_name = "LAI"          
os.makedirs(os.path.join(base,'tiffs',var_name),exist_ok = True)
out_tif = os.path.join(base,'tiffs',var_name,var_name+'.tiff')
nodata = -1e+32

ds = xr.open_dataset(nc_file, decode_times=False, engine='netcdf4')
da = ds[var_name]


evap = ds[var_name]    
pft_names = ds["NamePFT"].values  # strings

ntime, bands, nlat, nlon = da.shape

lat = da["lat"].values
lon = da["lon"].values

da = da.squeeze()
da = da.sortby("lat", ascending=True)

dlat = abs(lat[1] - lat[0])
dlon = abs(lon[1] - lon[0])


transform = from_origin(
    lon.min()-dlon/2,
    lat.max()+dlat/2,
    dlon,
    dlat
)


fill_value = da.attrs.get("_FillValue", nodata)


for t in range(evap.sizes["time"]):
    time_val = ds["time"].values[t]

    # out_file = out_dir / f"evap_time_{t:03d}.tif"
    out_tif = os.path.join(base,'tiffs',var_name,var_name+f"_time_{t:03d}.tif")
    # Data shape: (pft, lat, lon)
    if not os.path.exists(out_tif):
        data = evap.isel(time=t).values.astype(np.float32)
    
        with rasterio.open(
            out_tif,
            "w",
            driver="GTiff",
            height=nlat,
            width=nlon,
            count=bands,               
            dtype="float32",
            crs="""GEOGCS["WGS 84",
                DATUM["WGS_1984",
                    SPHEROID["WGS 84",6378137,298.257223563]],
                PRIMEM["Greenwich",0],
                UNIT["degree",0.0174532925199433]]""",
            transform=transform,
            nodata=nodata,
            compress="lzw"
        ) as dst:
        
        
            for i, pft_name in enumerate(pft_names):
                band = data[i, :, :]
                dst.write(band, i + 1)
                dst.set_band_description(i + 1, str(pft_name))
                
        with rasterio.open(out_tif) as src:
            data = src.read()      
            profile = src.profile
    
        data_flipped = np.flip(data, axis=1)
        
        with rasterio.open(out_tif, "w", **profile) as dst:
            dst.write(data_flipped)
    
    
        print(f"Written {out_tif}")


Written notebooks\..\tiffs\LAI\LAI_time_000.tif
Written notebooks\..\tiffs\LAI\LAI_time_001.tif
Written notebooks\..\tiffs\LAI\LAI_time_002.tif
Written notebooks\..\tiffs\LAI\LAI_time_003.tif
Written notebooks\..\tiffs\LAI\LAI_time_004.tif
Written notebooks\..\tiffs\LAI\LAI_time_005.tif
Written notebooks\..\tiffs\LAI\LAI_time_006.tif
Written notebooks\..\tiffs\LAI\LAI_time_007.tif
Written notebooks\..\tiffs\LAI\LAI_time_008.tif
Written notebooks\..\tiffs\LAI\LAI_time_009.tif
Written notebooks\..\tiffs\LAI\LAI_time_010.tif
Written notebooks\..\tiffs\LAI\LAI_time_011.tif
Written notebooks\..\tiffs\LAI\LAI_time_012.tif
Written notebooks\..\tiffs\LAI\LAI_time_013.tif
Written notebooks\..\tiffs\LAI\LAI_time_014.tif
Written notebooks\..\tiffs\LAI\LAI_time_015.tif
Written notebooks\..\tiffs\LAI\LAI_time_016.tif
Written notebooks\..\tiffs\LAI\LAI_time_017.tif
Written notebooks\..\tiffs\LAI\LAI_time_018.tif
Written notebooks\..\tiffs\LAI\LAI_time_019.tif
Written notebooks\..\tiffs\LAI\LAI_time_